# Extract/Transform Rust RFCs and PRs
This cell iterates through all Rust RFCs in the `rust-lang/rfcs` repository, from the very first RFC merged when the process began in 2014 onward, and the pull request that introduced each one. Unlike PEPs, RFCs have no standalone JSON index, so the RFC list is built from the `text/` directory of the repo itself (each RFC is a file named `NNNN-slug.md`), and the title / start date are parsed out of each file's own front matter.

Unlike the PEP <-> PR relationship (which can be many-to-many, since a PEP is amended by many PRs over its life), an RFC has a one-to-one relationship with the PR that introduced it, and every RFC file states that PR explicitly in its own front matter (`- RFC PR: [rust-lang/rfcs#NNNN]`). So instead of the PEP pipeline's approach of walking each file's full commit history through GraphQL (an expensive, deeply-nested query that would not scale well across the ~10x larger number of RFCs vs. PEPs), this pipeline reads that PR number directly out of the file text it already has to fetch for the title/date — an O(1) lookup per RFC. The costly commit-history walk is only used as a fallback for the small number of very early (2014-era) RFCs that predate this convention.

In [1]:
import json
import re
import time
from datetime import datetime
import requests

# 1. Load Configuration
with open("config.json") as f:
    config = json.load(f)

gh_token = config["github_token"]
headers = {
    "Authorization": f"Bearer {gh_token}",
    "Content-Type": "application/json",
}

REPO_OWNER = "rust-lang"
REPO_NAME = "rfcs"
# The RFC process itself began in Feb 2014 with RFC 0001, so this cut-off
# plays the same role as PEPs' TARGET_DATE but is expected to match ~all RFCs.
TARGET_DATE = datetime.fromisoformat("2014-01-01")


def get_matching_rfcs():
    """List every merged RFC text file in the repo (there is no RFC JSON index like PEPs has)."""
    tree_url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/git/trees/master?recursive=1"
    resp = requests.get(tree_url, headers=headers)
    resp.raise_for_status()
    tree_data = resp.json()

    # RFC text files live at text/NNNN-some-slug.md
    rfc_pattern = re.compile(r"^text/(\d{4})-(.+)\.md$")

    matching_rfcs = {}
    for entry in tree_data.get("tree", []):
        path = entry.get("path", "")
        match = rfc_pattern.match(path)
        if not match:
            continue
        num_int = int(match.group(1))
        slug = match.group(2)
        matching_rfcs[num_int] = {
            "formatted_id": f"rfc-{num_int:04d}",
            "filename": path.split("/")[-1],
            # Fallback title derived from the filename slug; refined below
            # once we pull the file's own "# Title" heading, when present.
            "title": slug.replace("-", " ").title(),
            "created_date": None,  # filled in below from the "Start Date" front matter, when present
        }
    return matching_rfcs


def chunk_list(lst, size):
    """Split a list into smaller chunks."""
    for i in range(0, len(lst), size):
        yield lst[i : i + size]


TITLE_RE = re.compile(r"^#\s+(.+)$", re.MULTILINE)
START_DATE_RE = re.compile(r"Start Date[:*]*\s*`?(\d{4}-\d{2}-\d{2})`?", re.IGNORECASE)
# The RFC's own front matter names the PR that introduced it, e.g.
# "- RFC PR: [rust-lang/rfcs#1216](https://github.com/rust-lang/rfcs/pull/1216)"
RFC_PR_RE = re.compile(r"RFC\s*PR[:*]*.*?#(\d+)", re.IGNORECASE)

# Blob-only batches are cheap (no nested GraphQL connections), so we can
# afford a larger batch size than the PEP pipeline's commit-history queries.
BLOB_CHUNK_SIZE = 50
# The commit-history fallback re-introduces a nested connection
# (history -> associatedPullRequests), which is the expensive part of the
# original PEP-style query, so it keeps a smaller, PEP-sized batch.
HISTORY_CHUNK_SIZE = 30


def fetch_rfc_blobs_graphql(rfc_items):
    """Fetch just the file text for multiple RFCs in a single, cheap GraphQL batch.

    This alone is enough to pull the title, start date, and (for almost every
    RFC) the introducing PR number straight out of the file's own front
    matter -- no commit-history walk required.
    """
    query_parts = []
    for rfc_num, info in rfc_items:
        alias = f"rfc_{rfc_num}"
        filename = info["filename"]
        query_parts.append(
            f"""
            {alias}: repository(owner: "{REPO_OWNER}", name: "{REPO_NAME}") {{
              object(expression: "HEAD:text/{filename}") {{
                ... on Blob {{ text }}
              }}
            }}
            """
        )

    gql_query = "query {" + "\n".join(query_parts) + "}"

    url = "https://api.github.com/graphql"
    resp = requests.post(url, headers=headers, json={"query": gql_query})
    resp.raise_for_status()
    data = resp.json().get("data", {})

    texts = {}
    for rfc_num, info in rfc_items:
        alias = f"rfc_{rfc_num}"
        repo_obj = data.get(alias) or {}
        blob = repo_obj.get("object") or {}
        texts[rfc_num] = blob.get("text") or ""
    return texts


def fetch_pr_via_history_graphql(rfc_items):
    """Fallback: walk commit history to find the introducing PR.

    Only used for the small set of legacy RFCs (mostly from 2014) that
    predate the "RFC PR" front-matter convention, so this expensive,
    nested-connection query never has to run against the full RFC corpus.
    """
    query_parts = []
    for rfc_num, info in rfc_items:
        alias = f"rfc_{rfc_num}"
        filename = info["filename"]
        query_parts.append(
            f"""
            {alias}: repository(owner: "{REPO_OWNER}", name: "{REPO_NAME}") {{
              object(expression: "HEAD") {{
                ... on Commit {{
                  history(path: "text/{filename}", first: 50) {{
                    nodes {{
                      associatedPullRequests(first: 5) {{
                        nodes {{ number }}
                      }}
                    }}
                  }}
                }}
              }}
            }}
            """
        )

    gql_query = "query {" + "\n".join(query_parts) + "}"

    url = "https://api.github.com/graphql"
    resp = requests.post(url, headers=headers, json={"query": gql_query})
    resp.raise_for_status()
    data = resp.json().get("data", {})

    results = {}
    for rfc_num, info in rfc_items:
        alias = f"rfc_{rfc_num}"
        repo_obj = data.get(alias) or {}
        pr_numbers = set()
        history_nodes = (repo_obj.get("object") or {}).get("history", {}).get("nodes", [])
        for commit in history_nodes:
            prs = commit.get("associatedPullRequests", {}).get("nodes", [])
            for pr in prs:
                pr_numbers.add(pr["number"])
        # Earliest PR touching the file's history is the best guess for the one that introduced it
        results[rfc_num] = min(pr_numbers) if pr_numbers else None
    return results


# Pipeline Execution
start_time = time.time()

matching_rfcs = get_matching_rfcs()
rfc_numbers = sorted(matching_rfcs.keys())
print(f"Found {len(matching_rfcs)} RFC files in the repository.\n")

rfc_pr_map = {}
needs_history_lookup = []

print("Fetching RFC file text (title, start date, RFC PR) via GraphQL...")
for chunk in chunk_list(rfc_numbers, BLOB_CHUNK_SIZE):
    chunk_items = [(num, matching_rfcs[num]) for num in chunk]
    texts = fetch_rfc_blobs_graphql(chunk_items)

    for rfc_num in chunk:
        text = texts.get(rfc_num, "")
        info = matching_rfcs[rfc_num]
        if not text:
            needs_history_lookup.append(rfc_num)
            continue

        title_match = TITLE_RE.search(text)
        if title_match:
            info["title"] = title_match.group(1).strip().strip("`")

        date_match = START_DATE_RE.search(text)
        if date_match:
            info["created_date"] = date_match.group(1)

        pr_match = RFC_PR_RE.search(text)
        if pr_match:
            rfc_pr_map[rfc_num] = [int(pr_match.group(1))]
        else:
            # No explicit "RFC PR" field -- almost always an early 2014-era RFC
            needs_history_lookup.append(rfc_num)

if needs_history_lookup:
    print(
        f"{len(needs_history_lookup)} RFC(s) had no explicit 'RFC PR' field; "
        "falling back to a commit-history lookup for just these..."
    )
    for chunk in chunk_list(needs_history_lookup, HISTORY_CHUNK_SIZE):
        chunk_items = [(num, matching_rfcs[num]) for num in chunk]
        fallback_results = fetch_pr_via_history_graphql(chunk_items)
        for rfc_num, pr_num in fallback_results.items():
            rfc_pr_map[rfc_num] = [pr_num] if pr_num is not None else []


def passes_cutoff(info):
    """Keep an RFC unless its parsed Start Date is before the 2014 cut-off."""
    if not info["created_date"]:
        return True
    try:
        return datetime.strptime(info["created_date"], "%Y-%m-%d") >= TARGET_DATE
    except ValueError:
        return True


matching_rfcs = {num: info for num, info in matching_rfcs.items() if passes_cutoff(info)}
rfc_pr_map = {num: prs for num, prs in rfc_pr_map.items() if num in matching_rfcs}

# Print Summary Output
print("\n" + "=" * 65)
print(f"{'RFC ID':<12} | {'Start Date':<12} | {'Introducing PR'}")
print("=" * 65)

total_prs_found = 0
for rfc_num, info in sorted(matching_rfcs.items()):
    pr_list = rfc_pr_map.get(rfc_num, [])
    total_prs_found += len(pr_list)

    pr_str = f"#{pr_list[0]}" if pr_list else "None"
    print(f"{info['formatted_id']:<12} | {str(info['created_date']):<12} | {pr_str}")

elapsed = round(time.time() - start_time, 2)
print("=" * 65)
print(f"Total matching RFCs: {len(matching_rfcs)}")
print(f"RFCs with a mapped PR: {total_prs_found}")
print(f"Execution time:      {elapsed} seconds")


Found 641 RFC files in the repository.

Fetching RFC file text (title, start date, RFC PR) via GraphQL...
23 RFC(s) had no explicit 'RFC PR' field; falling back to a commit-history lookup for just these...

RFC ID       | Start Date   | Introducing PR
rfc-0001     | 2014-03-11   | #1
rfc-0002     | 2014-03-11   | #2
rfc-0008     | 2014-03-14   | #8
rfc-0016     | 2014-03-20   | #16
rfc-0019     | 2014-09-18   | #19
rfc-0026     | 2014-03-31   | #26
rfc-0034     | 2014-04-05   | #34
rfc-0040     | 2014-04-08   | #40
rfc-0042     | 2014-04-12   | #42
rfc-0048     | 2014-06-10   | #48
rfc-0049     | 2014-03-20   | #49
rfc-0050     | 2014-04-18   | #50
rfc-0059     | 2014-04-30   | #59
rfc-0060     | 2014-04-30   | #60
rfc-0063     | 2014-05-02   | #63
rfc-0066     | 2014-05-04   | #66
rfc-0068     | 2014-06-11   | #68
rfc-0069     | 2014-05-05   | #69
rfc-0071     | 2014-05-07   | #71
rfc-0079     | 2014-05-17   | #79
rfc-0085     | 2014-05-21   | #85
rfc-0086     | 2014-05-22   | #86
rfc

# Load RFCs and PRs
This cell inserts the results of keying RFCs to PRs into a MongoDB collection.

In [ ]:
from datetime import datetime
from pymongo import MongoClient
import tiktoken

# 1. Setup MongoDB Connection & Tokenizer
mongo_connection_string = config["mongo_uri"]
client = MongoClient(mongo_connection_string)
db = client.consensus
collection = db.rfc_prs

tokenizer = tiktoken.get_encoding("cl100k_base")

def store_rfc_pr_mapping(rfc_num, info, pr_list):
    """Store RFC to PR mapping in MongoDB matching the specified schema pattern."""
    formatted_id = info["formatted_id"]
    title = info["title"]
    pr_str = ", ".join(f"#{pr}" for pr in pr_list) if pr_list else "None"

    # Construct text string for token count tracking
    text = f"RFC {rfc_num} ({formatted_id}): {title}. PRs: {pr_str}"
    tokens = tokenizer.encode(text)

    doc = {
        "source": "github_rfcs",
        "url": f"https://github.com/{REPO_OWNER}/{REPO_NAME}/blob/master/text/{info['filename']}",
        "repo": f"{REPO_OWNER}/{REPO_NAME}",
        "type": "rfc_pr_mapping",
        "rfc_number": rfc_num,
        "formatted_id": formatted_id,
        "title": title,
        "created_date": info["created_date"],
        "pr_numbers": pr_list,
        "pr_count": len(pr_list),
        "raw_text": text,
        "tokens": tokens,
        "metadata": {
            "fetched_at": datetime.utcnow().isoformat(),
        },
    }

    # Upsert document using rfc_number to safely allow re-running without duplicate keys
    collection.update_one({"rfc_number": rfc_num}, {"$set": doc}, upsert=True)


# 2. Iterate through mapping results from previous cell and insert into MongoDB
print(f"Inserting {len(matching_rfcs)} records into MongoDB collection '{collection.name}'...")

inserted_count = 0
for rfc_num, info in sorted(matching_rfcs.items()):
    pr_list = rfc_pr_map.get(rfc_num, [])
    store_rfc_pr_mapping(rfc_num, info, pr_list)
    inserted_count += 1

print(f"Successfully stored/updated {inserted_count} documents in MongoDB!")


# Extract/Load Comments
This cell fetches comments from each pull request and loads the comments into a document, which contains both keys for both the PR and RFC. No transformation procedure occurs here, as the transformation involves textual coding to be handled in a separate pipeline.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import hashlib
import json
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
from pymongo import MongoClient, UpdateOne
import tiktoken


def generate_author_pseudonym(author_username):
    """Generate a consistent SHA256 hash pseudonym for an author username."""
    if not author_username or author_username == "ghost":
        return "user_anonymous"

    # Compute plain SHA256 hash
    hash_bytes = hashlib.sha256(
        author_username.lower().strip().encode("utf-8")
    ).hexdigest()

    # Truncate to 12 hex characters for readable identifiers (e.g. user_8f4a1c09b2e1)
    return f"user_{hash_bytes[:12]}"


# 1. Cap MongoClient pool size to fit within Free Tier (Max 100 connections)
mongo_connection_string = config["mongo_uri"]
client = MongoClient(mongo_connection_string, maxPoolSize=10)
db = client.consensus

mappings_collection = db.rfc_prs
comments_collection = db.rfc_pr_comments

tokenizer = tiktoken.get_encoding("cl100k_base")

# 2. Setup GitHub HTTP Session
session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {gh_token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
})

retries = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504, 429])
adapter = HTTPAdapter(max_retries=retries, pool_connections=20, pool_maxsize=20)
session.mount("https://", adapter)

# 3. Load Mappings from MongoDB
print(f"Loading RFC-PR mappings from '{mappings_collection.name}'...")

pr_to_rfcs = {}
for doc in mappings_collection.find({}, {"rfc_number": 1, "pr_numbers": 1}):
    rfc_num = doc.get("rfc_number")
    pr_list = doc.get("pr_numbers", [])
    for pr_num in pr_list:
        if pr_num not in pr_to_rfcs:
            pr_to_rfcs[pr_num] = []
        if rfc_num and rfc_num not in pr_to_rfcs[pr_num]:
            pr_to_rfcs[pr_num].append(rfc_num)

all_pr_numbers = sorted(list(pr_to_rfcs.keys()))
print(f"Found {len(all_pr_numbers)} unique PRs.")


def fetch_paginated_endpoint(url):
    """Fetch paginated items from GitHub API."""
    items = []
    page = 1

    while True:
        resp = session.get(url, params={"per_page": 100, "page": page})

        if resp.status_code == 403:
            reset_time = int(resp.headers.get("X-RateLimit-Reset", time.time() + 60))
            time.sleep(max(reset_time - int(time.time()), 5))
            continue

        if not resp.ok:
            break

        data = resp.json()
        if not data:
            break

        items.extend(data)
        if "next" not in resp.links:
            break
        page += 1

    return items


def fetch_all_comments_for_pr(pr_number):
    """Fetch review & issue comments for a given PR."""
    review_comments_url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/pulls/{pr_number}/comments"
    issue_comments_url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/issues/{pr_number}/comments"

    review_comments = fetch_paginated_endpoint(review_comments_url)
    issue_comments = fetch_paginated_endpoint(issue_comments_url)

    processed_comments = []
    for c in review_comments:
        processed_comments.append((c, "pull_request_review_comment"))
    for c in issue_comments:
        processed_comments.append((c, "pull_request_issue_comment"))

    return pr_number, processed_comments


def prepare_bulk_doc(comment_data, pr_number, associated_rfcs):
    """Format comment into a lightweight, IRB-compliant MongoDB BSON document."""
    comment, comment_type = comment_data
    text = comment.get("body", "") or ""

    # Store token COUNT instead of full array of integers to save 50%+ disk space
    token_count = len(tokenizer.encode(text))

    # Plain SHA256 Pseudonymization
    raw_author = (
        comment.get("user", {}).get("login") if comment.get("user") else "ghost"
    )
    pseudonymized_author = generate_author_pseudonym(raw_author)

    comment_id = comment["id"]

    doc = {
        "source": "github",
        "url": comment.get("html_url", ""),
        "repo": f"{REPO_OWNER}/{REPO_NAME}",
        "type": comment_type,
        "pr_number": pr_number,
        "rfc_numbers": associated_rfcs,
        "author_id": pseudonymized_author,  # Pseudonymized handle
        "created_at": comment.get("created_at"),
        "raw_text": text,
        "token_count": token_count,
        "metadata": {
            "comment_id": comment_id,
            "in_reply_to_id": comment.get("in_reply_to_id"),
        },
    }

    # Single upsert keyed strictly on unique comment_id
    return UpdateOne(
        {"metadata.comment_id": comment_id},
        {"$set": doc},
        upsert=True,
    )


# 4. Fetch comments in parallel
print(f"Fetching review & issue comments for {len(all_pr_numbers)} PRs...")
start_time = time.time()

bulk_operations = []
total_comments = 0

with ThreadPoolExecutor(max_workers=15) as executor:
    futures = {
        executor.submit(fetch_all_comments_for_pr, pr_num): pr_num
        for pr_num in all_pr_numbers
    }

    for future in as_completed(futures):
        pr_number, comments = future.result()
        total_comments += len(comments)
        associated_rfcs = pr_to_rfcs.get(pr_number, [])
        for c in comments:
            bulk_operations.append(prepare_bulk_doc(c, pr_number, associated_rfcs))

# 5. Bulk Insert in Batches
if bulk_operations:
    print(f"Executing bulk MongoDB write for {len(bulk_operations)} comments...")

    BATCH_SIZE = 1000
    for i in range(0, len(bulk_operations), BATCH_SIZE):
        batch = bulk_operations[i : i + BATCH_SIZE]
        comments_collection.bulk_write(batch)

elapsed = round(time.time() - start_time, 2)
print(
    f"Done! Processed {len(all_pr_numbers)} PRs, stored {total_comments} comments in {elapsed} seconds."
)
